<a href="https://colab.research.google.com/github/juanitarhea/ai-drone-swarm/blob/main/M1_SwarmNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 16.9 MB/s eta 0:00:00


In [ ]:
import torch
from torch_geometric.data import Data

# Node features: [x, y, battery, task_status]
x = torch.tensor([
    [0.1, 0.2, 0.9, 0],
    [0.4, 0.2, 0.6, 1],
    [0.3, 0.8, 0.3, 2],
    [0.9, 0.5, 0.7, 0],
    [0.7, 0.1, 0.2, 1],
], dtype=torch.float)

# Connections (who talks to who)
edge_index = torch.tensor([
    [0,1,2,3,4, 1,2],
    [1,0,3,2,3, 4,0]
], dtype=torch.long)

# Labels (correct task per drone)
y = torch.tensor([0, 1, 2, 0, 1])  # 0=search,1=deliver,2=map

data = Data(x=x, edge_index=edge_index, y=y)

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class SwarmNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(4, 64)
        self.conv2 = GCNConv(64, 3)

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

In [ ]:
model = SwarmNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = torch.nn.CrossEntropyLoss()

for epoch in range(100):
    model.train()
    optimizer.zero_grad()

    out = model(data)
    loss = loss_fn(out, data.y)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 1.1384142637252808
Epoch 10, Loss: 0.9486533403396606
Epoch 20, Loss: 0.7937493324279785
Epoch 30, Loss: 0.617153525352478
Epoch 40, Loss: 0.4767185151576996
Epoch 50, Loss: 0.3708670735359192
Epoch 60, Loss: 0.29126983880996704
Epoch 70, Loss: 0.22959327697753906
Epoch 80, Loss: 0.1807766854763031
Epoch 90, Loss: 0.14211392402648926


In [ ]:
model.eval()
pred = model(data).argmax(dim=1)
print("Predicted tasks:", pred)

Predicted tasks: tensor([0, 1, 2, 0, 1])


In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score, f1_score, classification_report
import time

# ── Generate realistic swarm data ────────────────────────────────────────────
torch.manual_seed(42)
num_drones = 20

# Features: [x_pos, y_pos, battery_level, has_camera]
x = torch.tensor([
    [0.1, 0.2, 0.9, 1.0],  # high battery, has camera → Search
    [0.5, 0.8, 0.8, 1.0],  # high battery, has camera → Search
    [0.3, 0.4, 0.7, 1.0],  # high battery, has camera → Search
    [0.9, 0.1, 0.6, 1.0],  # med battery, has camera  → Search
    [0.2, 0.9, 0.5, 1.0],  # med battery, has camera  → Search
    [0.4, 0.3, 0.4, 1.0],  # med battery, has camera  → Search
    [0.7, 0.6, 0.9, 0.0],  # high battery, no camera  → Deliver
    [0.8, 0.2, 0.8, 0.0],  # high battery, no camera  → Deliver
    [0.1, 0.7, 0.7, 0.0],  # high battery, no camera  → Deliver
    [0.6, 0.5, 0.6, 0.0],  # med battery, no camera   → Deliver
    [0.3, 0.8, 0.5, 0.0],  # med battery, no camera   → Deliver
    [0.5, 0.1, 0.4, 0.0],  # med battery, no camera   → Deliver
    [0.9, 0.9, 0.2, 1.0],  # low battery, has camera  → Map
    [0.1, 0.1, 0.1, 1.0],  # low battery, has camera  → Map
    [0.7, 0.3, 0.2, 0.0],  # low battery, no camera   → Map
    [0.4, 0.6, 0.1, 0.0],  # low battery, no camera   → Map
    [0.6, 0.4, 0.3, 1.0],  # low battery, has camera  → Map
    [0.2, 0.5, 0.2, 0.0],  # low battery, no camera   → Map
    [0.8, 0.7, 0.1, 1.0],  # low battery, has camera  → Map
    [0.5, 0.5, 0.3, 0.0],  # low battery, no camera   → Map
], dtype=torch.float)

# Labels based on logical rules:
# Search = high battery + has camera (0)
# Deliver = any battery + no camera (1)
# Map = low battery (2)
true_labels = torch.tensor([
    0, 0, 0, 0, 0, 0,   # Search drones
    1, 1, 1, 1, 1, 1,   # Deliver drones
    2, 2, 2, 2, 2, 2, 2, 2  # Map drones
])

# Connect nearby drones as a mesh network
edge_index = torch.tensor([
    [0,1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,
     9,10,10,11,11,12,12,13,13,14,14,15,15,16,16,17,17,18,18,19,
     0,6,1,7,2,8,3,9,4,10,5,11],
    [1,0,2,1,3,2,4,3,5,4,6,5,7,6,8,7,9,8,
     10,9,11,10,12,11,13,12,14,13,15,14,16,15,17,16,18,17,19,18,
     6,0,7,1,8,2,9,3,10,4,11,5]
], dtype=torch.long)

# ── Model ────────────────────────────────────────────────────────────────────
class SwarmNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(4, 64)
        self.conv2 = GCNConv(64, 32)
        self.conv3 = GCNConv(32, 3)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = self.conv3(x, edge_index)
        return x

# ── Train ────────────────────────────────────────────────────────────────────
model = SwarmNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

start_time = time.time()
for epoch in range(500):
    model.train()
    optimizer.zero_grad()

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score, f1_score, classification_report
import time

# ── Generate realistic swarm data ────────────────────────────────────────────
torch.manual_seed(42)
num_drones = 20

# Features: [x_pos, y_pos, battery_level, has_camera]
x = torch.tensor([
    [0.1, 0.2, 0.9, 1.0],  # high battery, has camera → Search
    [0.5, 0.8, 0.8, 1.0],  # high battery, has camera → Search
    [0.3, 0.4, 0.7, 1.0],  # high battery, has camera → Search
    [0.9, 0.1, 0.6, 1.0],  # med battery, has camera  → Search
    [0.2, 0.9, 0.5, 1.0],  # med battery, has camera  → Search
    [0.4, 0.3, 0.4, 1.0],  # med battery, has camera  → Search
    [0.7, 0.6, 0.9, 0.0],  # high battery, no camera  → Deliver
    [0.8, 0.2, 0.8, 0.0],  # high battery, no camera  → Deliver
    [0.1, 0.7, 0.7, 0.0],  # high battery, no camera  → Deliver
    [0.6, 0.5, 0.6, 0.0],  # med battery, no camera   → Deliver
    [0.3, 0.8, 0.5, 0.0],  # med battery, no camera   → Deliver
    [0.5, 0.1, 0.4, 0.0],  # med battery, no camera   → Deliver
    [0.9, 0.9, 0.2, 1.0],  # low battery, has camera  → Map
    [0.1, 0.1, 0.1, 1.0],  # low battery, has camera  → Map
    [0.7, 0.3, 0.2, 0.0],  # low battery, no camera   → Map
    [0.4, 0.6, 0.1, 0.0],  # low battery, no camera   → Map
    [0.6, 0.4, 0.3, 1.0],  # low battery, has camera  → Map
    [0.2, 0.5, 0.2, 0.0],  # low battery, no camera   → Map
    [0.8, 0.7, 0.1, 1.0],  # low battery, has camera  → Map
    [0.5, 0.5, 0.3, 0.0],  # low battery, no camera   → Map
], dtype=torch.float)

# Labels based on logical rules:
# Search = high battery + has camera (0)
# Deliver = any battery + no camera (1)
# Map = low battery (2)
true_labels = torch.tensor([
    0, 0, 0, 0, 0, 0,   # Search drones
    1, 1, 1, 1, 1, 1,   # Deliver drones
    2, 2, 2, 2, 2, 2, 2, 2  # Map drones
])

# Connect nearby drones as a mesh network
edge_index = torch.tensor([
    [0,1,1,2,2,3,3,4,4,5,5,6,6,7,7,8,8,9,
     9,10,10,11,11,12,12,13,13,14,14,15,15,16,16,17,17,18,18,19,
     0,6,1,7,2,8,3,9,4,10,5,11],
    [1,0,2,1,3,2,4,3,5,4,6,5,7,6,8,7,9,8,
     10,9,11,10,12,11,13,12,14,13,15,14,16,15,17,16,18,17,19,18,
     6,0,7,1,8,2,9,3,10,4,11,5]
], dtype=torch.long)

# ── Model ────────────────────────────────────────────────────────────────────
class SwarmNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(4, 64)
        self.conv2 = GCNConv(64, 32)
        self.conv3 = GCNConv(32, 3)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = self.conv3(x, edge_index)
        return x

# ── Train ────────────────────────────────────────────────────────────────────
model = SwarmNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

start_time = time.time()
for epoch in range(500):
    model.train()
    optimizer.zero_grad()
    out = model(x, edge_index)
    loss = F.cross_entropy(out, true_labels)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

latency_ms = ((time.time() - start_time) / 500) * 1000

# ── Evaluate ─────────────────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    out = model(x, edge_index)
    predicted = out.argmax(dim=1)

y_true = true_labels.numpy()
y_pred = predicted.numpy()

accuracy = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred, average='weighted')

print("\n" + "=" * 45)
print("        SWARMNET EVALUATION RESULTS")
print("=" * 45)
print(f"  Task Allocation Accuracy : {accuracy * 100:.1f}%")
print(f"  Weighted F1-Score        : {f1:.4f}")
print(f"  Final Training Loss      : {loss.item():.4f}")
print(f"  Avg Allocation Latency   : {latency_ms:.2f} ms")
print(f"  Swarm Size Tested        : {num_drones} drones")
print("=" * 45)
print("\nPer-Task Breakdown:")
print(classification_report(y_true, y_pred,
      target_names=["Search", "Deliver", "Map"]))

print("Drone Task Assignments:")
task_names = ["Search", "Deliver", "Map"]
for i, task in enumerate(predicted):
    marker = "✅" if task.item() == y_true[i] else "❌"
    print(f"  Drone {i+1:2d} → {task_names[task.item()]} {marker}")

Epoch 0, Loss: 1.0969
Epoch 100, Loss: 0.1194
Epoch 200, Loss: 0.0200
Epoch 300, Loss: 0.0060
Epoch 400, Loss: 0.0028

        SWARMNET EVALUATION RESULTS
  Task Allocation Accuracy : 100.0%
  Weighted F1-Score        : 1.0000
  Final Training Loss      : 0.0016
  Avg Allocation Latency   : 6.09 ms
  Swarm Size Tested        : 20 drones

Per-Task Breakdown:
              precision    recall  f1-score   support

      Search       1.00      1.00      1.00         6
     Deliver       1.00      1.00      1.00         6
         Map       1.00      1.00      1.00         8

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20

Drone Task Assignments:
  Drone  1 → Search ✅
  Drone  2 → Search ✅
  Drone  3 → Search ✅
  Drone  4 → Search ✅
  Drone  5 → Search ✅
  Drone  6 → Search ✅
  Drone  7 → Deliver ✅
  Drone  8 → Deliver ✅
  Drone  9 → Deliver ✅
  Drone 10 → Deliver ✅
  Drone 11 

In [ ]:
def create_edges(positions, threshold=0.5):
    edges = []
    for i in range(len(positions)):
        for j in range(len(positions)):
            if i != j:
                dist = torch.dist(positions[i], positions[j])
                if dist < threshold:
                    edges.append([i, j])
    return torch.tensor(edges).t()

In [ ]:
import torch
from torch_geometric.nn import GCNConv

class SwarmNet(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(4, 64)   # 4 input features per drone
        self.conv2 = GCNConv(64, 3)   # 3 output tasks: search, deliver, map

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index)
        return x